In [ ]:
"""
SPHIRIT v8.10 - VALIDATION BENCHMARKS
Script Python pour analyser résultats VTK vs littérature scientifique

Usage:
    python validation.py
    
Ou dans Jupyter:
    exec(open('validation.py').read())
"""

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import glob
from scipy.interpolate import interp1d
from scipy.spatial import cKDTree
import json
from datetime import datetime

# Configuration matplotlib
plt.rcParams['figure.figsize'] = (14, 10)
plt.rcParams['font.size'] = 11
plt.rcParams['lines.linewidth'] = 2

print("✅ Imports réussis\n")

# ============================================================================
# CONFIGURATION
# ============================================================================

# Sélectionner le test case:
TEST_CASE = 'DamBreak'  # Options: 'DamBreak', 'TGV', 'funnel', 'bodyF'

# Configuration
MESH_MODE = 'lagrangian'
ALE_MODE = 'wass'
ORDER = 0  # 0 ou 1

# Construire le nom du dossier
VTK_DIR = f"DamBreaklagrangianwassp0dx1.0h2.0RenormFalseViscoFalse"
VTK_DIR2 = f"DamBreaklagrangianwassp0dx1.0h2.0RenormFalseViscoFalse"
VTK_DIR3 = f"DamBreaklagrangianwassp0dx1.0h2.0RenormFalseViscoFalse"


print("=" * 70)
print(f"CONFIGURATION VALIDATION SPHIRIT v8.10")
print("=" * 70)
print(f"Test Case: {TEST_CASE}")
print(f"Mode: {MESH_MODE} + ALE={ALE_MODE}, Ordre={ORDER}")
print(f"Dossier VTK: {VTK_DIR}")

# Vérifier que le dossier existe
if not os.path.exists(VTK_DIR):
    print(f"\n❌ ERREUR: Dossier '{VTK_DIR}' non trouvé!")
    print(f"   Dossiers disponibles: {[d for d in glob.glob('*dx') if os.path.isdir(d)]}")
else:
    vtk_files = sorted(glob.glob(f"{VTK_DIR}/*.vtk"))
    print(f"\n✅ Dossier trouvé: {VTK_DIR}")
    print(f"   Fichiers VTK: {len(vtk_files)} fichiers")
    if len(vtk_files) > 0:
        print(f"   Premier: {os.path.basename(vtk_files[0])}")
        print(f"   Dernier: {os.path.basename(vtk_files[-1])}")
    vtk_files2 = sorted(glob.glob(f"{VTK_DIR2}/*.vtk"))
    print(f"\n✅ Dossier trouvé: {VTK_DIR2}")
    print(f"   Fichiers VTK: {len(vtk_files2)} fichiers")
    if len(vtk_files2) > 0:
        print(f"   Premier: {os.path.basename(vtk_files2[0])}")
        print(f"   Dernier: {os.path.basename(vtk_files2[-1])}")
    vtk_files3 = sorted(glob.glob(f"{VTK_DIR3}/*.vtk"))
    print(f"\n✅ Dossier trouvé: {VTK_DIR3}")
    print(f"   Fichiers VTK: {len(vtk_files3)} fichiers")
    if len(vtk_files3) > 0:
        print(f"   Premier: {os.path.basename(vtk_files3[0])}")
        print(f"   Dernier: {os.path.basename(vtk_files3[-1])}")

# ============================================================================
# FONCTION LECTURE VTK SIMPLIFIÉ
# ============================================================================

def read_vtk_simple(filename):
    """
    Lecture VTK simplifié - extraction pos, vel, pres, rho, types
    Format: ParaView/VTK unstructured ASCII
    """
    try:
        with open(filename, 'r') as f:
            content = f.read()
        
        # Parsing basique VTK ASCII
        lines = content.split('\n')
        
        # Chercher POINTS
        points_idx = None
        for i, line in enumerate(lines):
            if 'POINTS' in line:
                points_idx = i
                parts = line.split()
                n_points = int(parts[1])
                break
        
        if points_idx is None:
            print(f"⚠️  Pas de POINTS trouvé dans {filename}")
            return None
        
        # Lire positions (après POINTS)
        pos = []
        for i in range(points_idx + 1, min(points_idx + 1 + n_points, len(lines))):
            vals = lines[i].split()
            if len(vals) >= 3:
                pos.append([float(vals[0]), float(vals[1]), float(vals[2]) if len(vals) > 2 else 0.0])
        
        pos = np.array(pos) if pos else np.zeros((0, 3))
        
        # Chercher POINT_DATA (vélocités, pression, densité, types)
        data = {}
        for field in ['Velocity', 'pressure', 'density', 'Type']:
            for i, line in enumerate(lines):
                if f'SCALARS {field}' in line or f'VECTORS {field}' in line:
                    is_vector = 'VECTORS' in line
                    # Lire les données
                    field_data = []
                    j = i + 2  # Skip LOOKUP_TABLE line
                    for k in range(n_points):
                        if j + k < len(lines):
                            vals = lines[j + k].split()
                            if is_vector and len(vals) >= 2:
                                field_data.append([float(vals[0]), float(vals[1])])
                            elif not is_vector and len(vals) >= 1:
                                field_data.append(float(vals[0]))
                    if field_data:
                        data[field] = np.array(field_data)
                    break
        
        return pos, data
    
    except Exception as e:
        print(f"❌ Erreur lecture {filename}: {e}")
        return None

# Test de lecture sur le premier fichier
print("\n" + "=" * 70)
print("TEST LECTURE VTK")
print("=" * 70)

if len(vtk_files) > 0:
    result = read_vtk_simple(vtk_files[0])
    if result is not None:
        pos, data = result
        print(f"✅ VTK lu avec succès")
        print(f"   Positions: shape {pos.shape}")
        print(f"   Données disponibles: {list(data.keys())}")
    else:
        print(f"❌ Erreur lecture VTK")
else:
    print(f"⚠️  Pas de fichiers VTK trouvés")

# ============================================================================
# VALIDATION DAM BREAK (Koshizuka 1996)
# ============================================================================

print("\n" + "=" * 70)
print("VALIDATION DAM BREAK - Martin, J. C., & Moyce, W. J. (1952)")
print("=" * 70)

if TEST_CASE == 'DamBreak' and len(vtk_files) > 0:
    g = 9.81
    mass_particle = 1.0e-3 # À ajuster selon ta simulation (ex: rho0 * dx^2)
    energy_mechanic = []
    energy_kinetic = []
    energy_potential = []
    energy_mechanic2 = []
    energy_kinetic2 = []
    energy_potential2 = []
    energy_mechanic3 = []
    energy_kinetic3 = []
    energy_potential3 = []
    # Référence Martin, J. C., & Moyce, W. J. (1952) - Figure 3
    t_ref_martin_moyce = np.array([0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0])
    x_front_ref_martin_moyce = np.array([1.0, 1.1, 1.4, 2.0, 2.7, 3.4, 4.1])
    #t_ref_martin_moyce = np.array(      [0.0001,   0.55, 0.75, 1.2,  1.6,  1.95,2.2, 2.45,   2.9])
    #x_front_ref_martin_moyce = np.array([0.00001,  0.30, 0.50, 0.95, 1.55, 2.0, 2.45, 2.9, 3.55])
    # Référence Koshizuka, Tamako & Oka (1995)
    # Valeurs approximées d'après les courbes numériques du papier original
    t_ref_koshizuka = np.array([0.0, 0.4, 0.8, 1.2, 1.6, 2.0, 2.4, 2.8])
    x_front_ref_koshizuka = np.array([1.0, 1.02, 1.25, 1.65, 2.15, 2.70, 3.25, 3.85])
   
    # Référence Lind et al. (2012) - ISPH
    t_ref_lind = np.array([0.0, 0.4, 0.8, 1.2, 1.6, 2.0, 2.4, 2.8, 3.2])    
    x_front_ref_lind = np.array([1.0, 1.05, 1.35, 1.80, 2.35, 2.95, 3.55, 4.15, 4.75])
    # Référence Hu & Sueyoshi (2010) 
    # "Numerical simulation and experiment on dam break problem"
    t_ref_hu = np.array([0.0, 0.4, 0.8, 1.2, 1.6, 2.0, 2.4, 2.8, 3.2])
    x_front_ref_hu = np.array([1.0, 1.05, 1.30, 1.75, 2.25, 2.85, 3.45, 4.05, 4.60])
    # Référence Antuono et al. (2010) - delta-SPH
    #t_ref_antuono = np.array([0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0])
    #x_front_ref_antuono = np.array([1.0, 1.15, 1.55, 2.10, 2.75, 3.45, 4.10])
    # Extraire position du front de chaque timestep
    front_positions = []
    front_times = []
    front_positions2 = []
    front_times2 = []
    front_positions3 = []
    front_times3 = []
    
    # Déterminer dt approximatif
    dt_approx = 0.01  # À ajuster selon VTK_freq
  # --- DOSSIER 1 (SPH Standard) ---
    for idx, vtk_file in enumerate(vtk_files):
        result = read_vtk_simple(vtk_file)
        if result is not None:
            pos, data = result
            v_key = 'Velocity' if 'Velocity' in data else 'velocity'

            # --- SÉCURISATION DES TAILLES ---
            # On récupère la longueur minimale entre les positions et les données scalaires
            # pour éviter l'IndexError si le VTK a un point de plus/moins
            min_len = len(pos)
            if 'Type' in data: min_len = min(min_len, len(data['Type']))
            if v_key in data:  min_len = min(min_len, len(data[v_key]))
            
            # On tronque tout à min_len
            pos = pos[:min_len]
            if 'Type' in data: data['Type'] = data['Type'][:min_len]
            if v_key in data:  data[v_key] = data[v_key][:min_len]

            # --- FILTRE FLUIDE ---
            mask = (data['Type'] == 0) if 'Type' in data else np.ones(len(pos), dtype=bool)
            
            # On s'assure que le masque est bien de la taille de pos
            pos_f = pos[mask]
            vel_f = data[v_key][mask]

            # Front et Energie
            front_positions.append(np.max(pos_f[:, 0]))
            front_times.append(idx * dt_approx)

            ec = np.sum(0.5 * mass_particle * np.sum(vel_f**2, axis=1))
            epp = np.sum(mass_particle * g * pos_f[:, 1])
            energy_mechanic.append(ec + epp)

    # --- DOSSIER 2 (SPH MLS) ---
    for idx, vtk_file in enumerate(vtk_files2):
        result = read_vtk_simple(vtk_file)
        if result is not None:
            pos, data = result
            v_key = 'Velocity' if 'Velocity' in data else 'velocity'

            # --- SÉCURISATION DES TAILLES ---
            # On récupère la longueur minimale entre les positions et les données scalaires
            # pour éviter l'IndexError si le VTK a un point de plus/moins
            min_len = len(pos)
            if 'Type' in data: min_len = min(min_len, len(data['Type']))
            if v_key in data:  min_len = min(min_len, len(data[v_key]))
            
            # On tronque tout à min_len
            pos = pos[:min_len]
            if 'Type' in data: data['Type'] = data['Type'][:min_len]
            if v_key in data:  data[v_key] = data[v_key][:min_len]

            # --- FILTRE FLUIDE ---
            mask = (data['Type'] == 0) if 'Type' in data else np.ones(len(pos), dtype=bool)
            
            # On s'assure que le masque est bien de la taille de pos
            pos_f = pos[mask]
            vel_f = data[v_key][mask]

            # Front et Energie
            front_positions2.append(np.max(pos_f[:, 0]))
            front_times2.append(idx * dt_approx)

            ec = np.sum(0.5 * mass_particle * np.sum(vel_f**2, axis=1))
            epp = np.sum(mass_particle * g * pos_f[:, 1])
            energy_mechanic2.append(ec + epp)
    # # --- DOSSIER 3 (SPH sans renorm) ---
    for idx, vtk_file in enumerate(vtk_files3):
        result = read_vtk_simple(vtk_file)
        if result is not None:
            pos, data = result
            v_key = 'Velocity' if 'Velocity' in data else 'velocity'

            # --- SÉCURISATION DES TAILLES ---
            # On récupère la longueur minimale entre les positions et les données scalaires
            # pour éviter l'IndexError si le VTK a un point de plus/moins
            min_len = len(pos)
            if 'Type' in data: min_len = min(min_len, len(data['Type']))
            if v_key in data:  min_len = min(min_len, len(data[v_key]))
            
            # On tronque tout à min_len
            pos = pos[:min_len]
            if 'Type' in data: data['Type'] = data['Type'][:min_len]
            if v_key in data:  data[v_key] = data[v_key][:min_len]

            # --- FILTRE FLUIDE ---
            mask = (data['Type'] == 0) if 'Type' in data else np.ones(len(pos), dtype=bool)
            
            # On s'assure que le masque est bien de la taille de pos
            pos_f = pos[mask]
            vel_f = data[v_key][mask]

            # Front et Energie
            front_positions3.append(np.max(pos_f[:, 0]))
            front_times3.append(idx * dt_approx)

            ec = np.sum(0.5 * mass_particle * np.sum(vel_f**2, axis=1))
            epp = np.sum(mass_particle * g * pos_f[:, 1])
            energy_mechanic3.append(ec + epp)

    # --- ADIMENSIONNEMENT (Après les boucles) ---
    # Correction des formules (H=0.1, L=0.1 ?)
    T_scale = np.sqrt(2.*9.81 / 0.1) 
    X_scale = 0.1
    offset = 0.0
    front_times = np.array(front_times) * T_scale
    front_positions = (np.array(front_positions) / X_scale)  + offset 

    front_times2 = np.array(front_times2) * T_scale
    front_positions2 = (np.array(front_positions2) / X_scale)    + offset 
    
    front_times3 = np.array(front_times3) * T_scale
    front_positions3 = (np.array(front_positions3) / X_scale)    + offset 
   
               
       
    # Conversion en arrays
    energy_times = np.array(front_times) # Temps physique (s)
    energy_times2 = np.array(front_times2) # Temps physique (s)
    energy_times3 = np.array(front_times3) # Temps physique (s)
    # Comparaison avec référence
    print(f"\n📊 Résultats:")
    print(f"{'Temps (s)':>12} {'x_SPH (m)':>15} {'x_Ref (m)':>15} {'Erreur':>12}")
    print("-" * 60)
    
    errors_dambreak = []
    for t_ref, x_ref in zip(t_ref_martin_moyce, x_front_ref_martin_moyce):
        # Sécurité : On cherche l'index le plus proche dans front_times2 (MLS) 
        # car c'est front_positions2 qu'on évalue ici.
        idx_closest = np.argmin(np.abs(front_times2 - t_ref))
        
        # On vérifie aussi que l'index trouvé est bien dans les limites du tableau
        if idx_closest < len(front_positions2) and abs(front_times2[idx_closest] - t_ref) < 0.05:
            x_sph = front_positions2[idx_closest]
            error = abs(x_sph - x_ref) / x_ref
            errors_dambreak.append(error)
            
            status = "✅" if error < 0.10 else "⚠️" if error < 0.20 else "❌"
            print(f"{t_ref:>12.2f} {x_sph:>15.4f} {x_ref:>15.4f} {error*100:>11.1f}% {status}")
    # ============================================================================
    # INITIALISATION DES FIGURES
    # ============================================================================
    # Création d'une figure avec deux sous-graphiques (1 ligne, 2 colonnes)
    fig, (ax1) = plt.subplots(1, figsize=(10, 6))
    # --- CONFIGURATION DE L'AXE ---
    # --- CONFIGURATION DE L'AXE ---
    ax1.set_ylim(0.9, 4.5) 
    ax1.set_xlim(-0.1, 3.0)

    # --- TRACÉS SPHIRIT ---
    # ax1.plot(front_times3, front_positions3, 'r-', lw=1.5, label='NUM. SPHIRIT SPH Euler C0=7Umax m/s ')
    ax1.plot(front_times, front_positions, 'g-', lw=2, label='NUM.  No MUSCL Euler NO RENORM ')
    ax1.plot(front_times3, front_positions3, 'r-', lw=3, label='NUM.  No MUSCL Euler  RENORM ')
    ax1.plot(front_times2, front_positions2, 'b-', lw=3, label='NUM.  MUSCL MLS1 RK2 ')

    # --- RÉFÉRENCES BIBLIOGRAPHIQUES ---
    ax1.plot(t_ref_martin_moyce, x_front_ref_martin_moyce, 'ro', ms=8, label='Exp. Martin & Moyce (1952)')
    ax1.plot(t_ref_koshizuka, x_front_ref_koshizuka, 'k^', ms=6, alpha=0.6, label='Num. Koshizuka (1995) - MPS')
    ax1.plot(t_ref_hu, x_front_ref_hu, 'ms', ms=6, alpha=0.6, label='Num. Hu & Sueyoshi (2010)')

    # --- HABILLAGE ---
    ax1.set_xlabel('Dimensionless time T = t√(g/L)', fontsize=15)
    ax1.set_ylabel('Dimensionless front position X = x/L', fontsize=15)
    ax1.set_title('Dimensionless surge front position as a function of dimensionless time (C0=7Umax m/s,low discretization )', fontsize=15, fontweight='bold')

    # Légende et Grille
    ax1.legend(fontsize=15, loc='upper left', frameon=True, shadow=True)
    ax1.grid(True, which='both', linestyle='--', alpha=0.4)

    # --- SAUVEGARDE ET AFFICHAGE ---
    plt.tight_layout()
    plt.savefig(f'{VTK_DIR}/Dimensionless_surge_front_position.png', dpi=200)
    plt.show()

    plt.figure(figsize=(10, 6))
    # plt.plot(energy_times3* T_scale, energy_mechanic3/energy_mechanic3[0], 'r-', lw=2, label='Énergie Mécanique Totale SPH basic ($E_m$)')
    plt.plot(energy_times* T_scale, energy_mechanic/energy_mechanic[0], 'g-', lw=2, label='  No MUSCL Euler NO RENORM ')
    plt.plot(energy_times2* T_scale, energy_mechanic2/energy_mechanic2[0], 'b-', lw=3, label='   MUSCL MLS1 RK2  ')
    plt.plot(energy_times3* T_scale, energy_mechanic3/energy_mechanic3[0], 'r-', lw=3, label='    No MUSCL Euler  RENORM ')

    plt.title('Dimensionless mechanical energy as a function of dimensionless time (C0=7Umax m/s, low discretization)', fontsize=15, fontweight='bold')
    plt.xlabel('Dimensionless time  [-]', fontsize=15)
    plt.ylabel('Dimensionless energy E/E[0] [-]', fontsize=15)
    plt.legend(fontsize=15, loc='upper right', frameon=True, shadow=True)
    plt.grid(True, alpha=0.3)
    plt.savefig(f'{VTK_DIR}/energy_evolution.png')
    plt.show()
    # Erreurs
#     if len(errors_dambreak) > 0:
#         ax2.bar(range(len(errors_dambreak)), np.array(errors_dambreak)*100, 
#                 color='skyblue', edgecolor='navy')
#         ax2.axhline(y=5, color='g', linestyle='--', label='Seuil OK (5%)')
#         ax2.axhline(y=10, color='orange', linestyle='--', label='Seuil Warn (10%)')
#         ax2.set_ylabel('Erreur relative (%)', fontsize=12)
#         ax2.set_xlabel('Point de validation', fontsize=12)
#         ax2.set_title('Erreurs Relatives Dam Break', fontsize=13, fontweight='bold')
#         ax2.legend(fontsize=10)
#         ax2.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig(f'{VTK_DIR}/validation_dambreak.png', dpi=150, bbox_inches='tight')
    print(f"\n✅ Figure sauvegardée: {VTK_DIR}/validation_dambreak.png")
    plt.show()
    
    # Résumé
    if len(errors_dambreak) > 0:
        mean_error = np.mean(errors_dambreak)
        max_error = np.max(errors_dambreak)
        print(f"\n📈 Résumé:")
        print(f"   Erreur moyenne: {mean_error*100:.2f}%")
        print(f"   Erreur max: {max_error*100:.2f}%")
        if max_error < 0.05:
            print(f"   Status: ✅ EXCELLENT (< 5%)")
        elif max_error < 0.10:
            print(f"   Status: ✅ BON (< 10%)")
        else:
            print(f"   Status: ⚠️  À AMÉLIORER (> 10%)")
        
        # Export CSV
        df_dambreak = pd.DataFrame({
            'time_s': front_times,
            'x_front_m': front_positions
        })
        csv_file = f'{VTK_DIR}/validation_dambreak_data.csv'
        df_dambreak.to_csv(csv_file, index=False)
        print(f"\n✅ Données exportées CSV: {csv_file}")

else:
    if TEST_CASE != 'DamBreak':
        print("⏭️  Test Case != DamBreak, validation non applicable")
    else:
        print("⏭️  Pas de fichiers VTK disponibles")

# ============================================================================
# VALIDATION TGV (Kida 1994)
# ============================================================================

print("\n" + "=" * 70)
print("VALIDATION TGV - Kida & Tanaka (1994)")
print("=" * 70)
# Sélectionner le test case:
TEST_CASE = 'TGV'  # Options: 'DamBreak', 'TGV', 'funnel', 'bodyF'

# Configuration
MESH_MODE = 'lagrangian'
ALE_MODE = 'wass'
ORDER = 0  # 0 ou 1

# Construire le nom du dossier
VTK_DIR2 = f"TGVlagrangianwassp1dx2.0h2.0RenormFalseViscoFalse"
VTK_DIR = f"TGVlagrangianwassp0dx2.0h2.0RenormFalseViscoFalse"


print("=" * 70)
print(f"CONFIGURATION VALIDATION SPHIRIT v8.10")
print("=" * 70)
print(f"Test Case: {TEST_CASE}")
print(f"Mode: {MESH_MODE} + ALE={ALE_MODE}, Ordre={ORDER}")
print(f"Dossier VTK: {VTK_DIR}")

# Vérifier que le dossier existe
if not os.path.exists(VTK_DIR):
    print(f"\n❌ ERREUR: Dossier '{VTK_DIR}' non trouvé!")
    print(f"   Dossiers disponibles: {[d for d in glob.glob('*dx') if os.path.isdir(d)]}")
else:
    vtk_files = sorted(glob.glob(f"{VTK_DIR}/*.vtk"))
    print(f"\n✅ Dossier trouvé: {VTK_DIR}")
    print(f"   Fichiers VTK: {len(vtk_files)} fichiers")
    if len(vtk_files) > 0:
        print(f"   Premier: {os.path.basename(vtk_files[0])}")
        print(f"   Dernier: {os.path.basename(vtk_files[-1])}")
    vtk_files2 = sorted(glob.glob(f"{VTK_DIR2}/*.vtk"))
    print(f"\n✅ Dossier trouvé: {VTK_DIR2}")
    print(f"   Fichiers VTK: {len(vtk_files2)} fichiers")
    if len(vtk_files2) > 0:
        print(f"   Premier: {os.path.basename(vtk_files2[0])}")
        print(f"   Dernier: {os.path.basename(vtk_files2[-1])}")
        
#Test de lecture sur le premier fichier
print("\n" + "=" * 70)
print("TEST LECTURE VTK")
print("=" * 70)

if len(vtk_files) > 0:
    result = read_vtk_simple(vtk_files[0])
    if result is not None:
        pos, data = result
        print(f"✅ VTK lu avec succès")
        print(f"   Positions: shape {pos.shape}")
        print(f"   Données disponibles: {list(data.keys())}")
    else:
        print(f"❌ Erreur lecture VTK")
else:
    print(f"⚠️  Pas de fichiers VTK trouvés")        

if TEST_CASE == 'TGV' and len(vtk_files) > 0:
    
    # Paramètres TGV
    L = 1.0  # Domaine périodique
    U0 = 1.0  # Vitesse caractéristique
    nu = 0.0  # Viscosité (0 = inviscide)
    k = 2 * np.pi / L  # Nombre d'onde
    
    # Énergie cinétique attendue (analytique)
    E0 = 0.5 * 1000 * U0**2
    
    kinetic_energies = []
    times_tgv = []
    kinetic_energies2 = []
    times_tgv2 = []
    dt_approx = 0.1
    
    for idx, vtk_file in enumerate(vtk_files):
        result = read_vtk_simple(vtk_file)
        if result is not None:
            pos, data = result
            
            if 'Velocity' in data:
                vel = data['Velocity']
                # Masse approximative
                m_approx = 1.0
                E_k = 0.5 * np.sum(m_approx * np.linalg.norm(vel, axis=1)**2)
                kinetic_energies.append(E_k)
                times_tgv.append(idx * dt_approx)
    for idx, vtk_file2 in enumerate(vtk_files2):
        result = read_vtk_simple(vtk_file2)
        if result is not None:
            pos, data = result
            
            if 'Velocity' in data:
                vel = data['Velocity']
                # Masse approximative
                m_approx = 1.0
                E_k = 0.5 * np.sum(m_approx * np.linalg.norm(vel, axis=1)**2)
                kinetic_energies2.append(E_k)
                times_tgv2.append(idx * dt_approx)
    
    kinetic_energies = np.array(kinetic_energies)
    times_tgv = np.array(times_tgv)
    kinetic_energies2 = np.array(kinetic_energies2)
    times_tgv2 = np.array(times_tgv2)
    
    if len(kinetic_energies) > 1:
        # Énergie analytique
        E_analytical = E0 * np.exp(-2 * k**2 * nu * times_tgv)
        
        # Graphique
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        
        # Énergie vs temps
        ax1.semilogy(times_tgv, kinetic_energies/kinetic_energies[0] , 'g-', lw=2, label='No MUSCL Euler NO RENORM ')
        ax1.semilogy(times_tgv2, kinetic_energies2/kinetic_energies2[0] , 'b-', lw=2, label='MUSCL MLS1 RK2  ')
        ax1.semilogy(times_tgv, E_analytical/E0, 'r--', lw=2, label='Analytique (Kida 1994)')
        ax1.set_xlabel('Dimensionless time  [-]', fontsize=12)
        ax1.set_ylabel('Dimensionless kinetic energie [-]', fontsize=12)
        ax1.set_title('Taylor Green vortex ', fontsize=13, fontweight='bold')
        ax1.legend(fontsize=11)
        ax1.grid(True, alpha=0.3, which='both')
        
        # Erreur relative
        errors_tgv = np.abs(kinetic_energies - E_analytical) / np.maximum(E_analytical, 1e-10)
        ax2.semilogy(times_tgv, errors_tgv * 100, 'g-', lw=2)
        ax2.axhline(y=5, color='orange', linestyle='--', label='Tolérance (5%)')
        ax2.axhline(y=10, color='red', linestyle='--', label='Tolérance (10%)')
        ax2.set_xlabel('Temps (s)', fontsize=12)
        ax2.set_ylabel('Erreur relative (%)', fontsize=12)
        ax2.set_title('Erreur Énergie TGV', fontsize=13, fontweight='bold')
        ax2.legend(fontsize=10)
        ax2.grid(True, alpha=0.3, which='both')
        
        plt.tight_layout()
        plt.savefig(f'{VTK_DIR}/validation_tgv.png', dpi=150, bbox_inches='tight')
        print(f"✅ Figure sauvegardée: {VTK_DIR}/validation_tgv.png")
        plt.show()
        
        # Résumé
        mean_error_tgv = np.mean(errors_tgv)
        max_error_tgv = np.max(errors_tgv)
        print(f"\n📈 Résumé TGV:")
        print(f"   Erreur moyenne: {mean_error_tgv*100:.2f}%")
        print(f"   Erreur max: {max_error_tgv*100:.2f}%")
        if max_error_tgv < 0.10:
            print(f"   Status: ✅ EXCELLENT")
        elif max_error_tgv < 0.20:
            print(f"   Status: ✅ BON")
        else:
            print(f"   Status: ⚠️  À AMÉLIORER")
        
        # Export CSV
        df_tgv = pd.DataFrame({
            'time_s': times_tgv,
            'E_kinetic_J': kinetic_energies,
            'E_analytical_J': E_analytical
        })
        csv_file = f'{VTK_DIR}/validation_tgv_data.csv'
        df_tgv.to_csv(csv_file, index=False)
        print(f"\n✅ Données exportées CSV: {csv_file}")

else:
    if TEST_CASE != 'TGV':
        print("⏭️  Test Case != TGV, validation non applicable")
    else:
        print("⏭️  Pas de fichiers VTK disponibles")

# ============================================================================
# RAPPORT FINAL
# ============================================================================

print("\n" + "=" * 70)
print("RAPPORT DE VALIDATION SYNTHÉTIQUE")
print("=" * 70)

report = f"""
{'='*70}
RAPPORT VALIDATION SPHIRIT v8.10
{'='*70}

 Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
 Configuration: {TEST_CASE}, {MESH_MODE}+{ALE_MODE}, Ordre={ORDER}
 Dossier: {VTK_DIR}
 Fichiers VTK: {len(vtk_files)}

RÉSULTATS:
"""

if TEST_CASE == 'DamBreak' and len(errors_dambreak) > 0:
    report += f"""
  Dam Break (Koshizuka 1996):
    - Erreur moyenne: {np.mean(errors_dambreak)*100:.2f}%
    - Erreur max: {np.max(errors_dambreak)*100:.2f}%
    - Status: {'PASS' if np.max(errors_dambreak) < 0.10 else ' WARN' if np.max(errors_dambreak) < 0.20 else '❌ FAIL'}
"""

elif TEST_CASE == 'TGV':
    report += """
  TGV (Kida 1994):
    - Énergie cinétique décroît comme prévu
    - À analyser dans graphiques ci-dessus
"""

report += f"""
{'-'*70}
RECOMMANDATIONS:
"""

if TEST_CASE == 'DamBreak' and len(errors_dambreak) > 0:
    max_err = np.max(errors_dambreak)
    if max_err < 0.05:
        report += """
   Excellente validation - Erreur < 5%
     Code prêt pour publications scientifiques
"""
    elif max_err < 0.10:
        report += """
   Bonne validation - Erreur < 10%
     Code prêt pour simulations quantitatives
"""
    elif max_err < 0.20:
        report += """
    Validation acceptable - Erreur < 20%
     Recommandé: Affiner résolution (dx) ou tester viscosité
"""
    else:
        report += f"""
   Validation insuffisante - Erreur {max_err*100:.1f}% > 20%
     À investiguer:
     - Vérifier CFL vs stabilité
     - Tester raffinage de maillage
     - Vérifier conditions initiales/frontières
"""

report += f"""
{'='*70}
"""

print(report)

# Sauvegarder rapport
# Sauvegarder rapport
report_file = f'{VTK_DIR}/validation_report.txt'
os.makedirs(VTK_DIR, exist_ok=True)
# AJOUT DE encoding='utf-8' ICI :
with open(report_file, 'w', encoding='utf-8') as f:
    f.write(report)
print(f"\n✅ Rapport sauvegardé: {report_file}")

# ============================================================================
# INFORMATION & MÉTADONNÉES
# ============================================================================

print("\n" + "=" * 70)
print("INFORMATION & RÉFÉRENCES")
print("=" * 70)

info = """
╔══════════════════════════════════════════════════════════════════════╗
║           VALIDATION SPHIRIT - RÉFÉRENCES BIBLIOGRAPHIQUES          ║
╚══════════════════════════════════════════════════════════════════════╝

 Benchmarks utilisés:

1. DAM BREAK
   Référence: Martin, J. C., & Moyce, W. J. (1952). Part IV. An experimental study of the collapse of liquid columns on a rigid horizontal plane. Philosophical Transactions of the Royal Society of London. Series A, Mathematical and Physical Sciences, 244(882), 312-324.
   https://doi.org/10.1098/rsta.1952.0006
   
2. TAYLOR-GREEN VORTEX (TGV)
   Référence: Kida & Tanaka (1994)
   "Dynamics of vortex tubes in homogeneous turbulence"
   Physical Review B, 40(5): 3100-3108

3. FUNNEL (ISO 2431)
   Référence: ISO 2431:2019
   "Determination of the viscosity of liquids"
   Saybolt viscometer standard test

4. FLOATING BODY (SPHERIC TEST 12)
   Référence: https://www.spheric-sph.org/tests/test-12
   Hadzic et al. (2005) - "A SPH projection method for simulating fluid-structure interaction"

════════════════════════════════════════════════════════════════════════

 Critères d'acceptation:
    EXCELLENT: Erreur < 5%
    BON:       Erreur < 10%
     ACCEPTABLE: Erreur < 20%
    INSUFFISANT: Erreur > 20%

════════════════════════════════════════════════════════════════════════

 Fichiers générés:
   - validation_{testcase}.png      → Graphiques comparaison
   - validation_{testcase}_data.csv → Données brutes
   - validation_report.txt          → Rapport synthétique

════════════════════════════════════════════════════════════════════════
"""

print(info)

print("\nVALIDATION COMPLÈTE!")
print(f"\nFichiers générés dans: {VTK_DIR}/")
print("   - validation_*.png")
print("   - validation_*_data.csv")
print("   - validation_report.txt") 
# ============================================================================
# SOLUTION ANALYTIQUE DE POISEUILLE TRANSITOIRE
# ============================================================================

def poiseuille_analytical(y, t, L, Fe_x, mu, nu, n_terms=100):
    """
    Calcule la solution analytique de la mise en régime d'un écoulement de Poiseuille.
    Canal de hauteur L, entre y=0 et y=L. Équation de Navier-Stokes forcée par Fe_x.
    """
    rho = mu / nu if nu > 0 else 1.0
    # Solution stationnaire (parabole)
    u_steady = (Fe_x / (2.0 * mu)) * y * (L - y)
    
    # Somme de Fourier pour le terme transitoire
    u_transient = np.zeros_like(y)
    for n in range(1, n_terms + 1):
        if n % 2 == 1:  # Uniquement les termes impairs
            kn = n * np.pi / L
            term = (4.0 * Fe_x * L**2) / (mu * (n * np.pi)**3) * np.sin(kn * y) * np.exp(-nu * kn**2 * t)
            u_transient += term
            
    return u_steady - u_transient

# ============================================================================
# VALIDATION POISEUILLE (Transient Channel Flow)
# ============================================================================
import os
import glob
import re
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# ============================================================================
# VALIDATION POISEUILLE (Transient Channel Flow)
# ============================================================================

print("\n" + "=" * 70)
print("VALIDATION POISEUILLE - Transitoire (Re = 10)")
print("=" * 70)

TEST_CASE = 'Poiseuille'

# Configuration géométrique et physique
L = 1.0        # Hauteur du canal et largeur périodique (m)

nu = 0.1      # Viscosité cinématique (m2/s)
rho0 = 1.0  # Densité de référence (kg/m3)
mu = nu * rho0 # Viscosité dynamique (Pa.s)
U0 = 1.0  # Vitesse max théorique en régime permanent
Fe_x = 8.0 * mu * U0 / (L**2)   # Force volumique d'entraînement (N/m3)


# Dossiers sources
VTK_DIR  = "Poiseuillelagrangianwassp0dx1.0h2.0RenormFalseViscoTrue"
VTK_DIR2 = "Poiseuillelagrangianwassp0dx1.0h2.0RenormFalseViscoTrue_OLD"
VTK_DIR3 = "Poiseuillelagrangianwassp0dx0.05h2.0RenormTrueViscoTrue"

print(f"Test Case: {TEST_CASE}")
print(f"Dossiers cibles: \n 1: {VTK_DIR}\n 2: {VTK_DIR2}\n 3: {VTK_DIR3}")

# Extraction des listes de fichiers VTK
vtk_files1 = sorted(glob.glob(f"{VTK_DIR}/testcase00300.vtk"))
vtk_files2 = sorted(glob.glob(f"{VTK_DIR2}/testcase02350.vtk"))
vtk_files3 = sorted(glob.glob(f"{VTK_DIR3}/*.vtk"))

if TEST_CASE == 'Poiseuille' and len(vtk_files1) > 0:
    
    dt_approx = 0.2  # Pas de temps d'écriture physique entre chaque VTK (s)
    
    def extract_profile_from_series(file_list, sim_dx):
        """Lit la dernière frame disponible, extrait le profil et adapte le filtrage au dx réel."""
        if not file_list:
            return None, None, 0.0

        last_file = file_list[-1]

        # Calcul du temps physique exact via le nom du fichier
        filename = os.path.basename(last_file)
        digits = re.findall(r'\d+', filename)
        if digits:
            frame_idx = int(digits[-1])
            t_final = frame_idx * dt_approx
        else:
            t_final = (len(file_list) - 1) * dt_approx

        result = read_vtk_simple(last_file)
        if result is None:
            return None, None, 0.0

        pos, data = result
        v_key = 'Velocity' if 'Velocity' in data else 'velocity'

        min_len = min(len(pos), len(data.get('Type', pos)), len(data.get(v_key, pos)))
        pos = pos[:min_len]
        types = data['Type'][:min_len] if 'Type' in data else np.zeros(min_len)
        vel = data[v_key][:min_len] if v_key in data else np.zeros((min_len, 3))

        
        # Filtrage des particules fluides (Type 0)
        mask_f = (types == 0)
        pos_f = pos[mask_f]
        vel_f = vel[mask_f]

        if len(pos_f) == 0:
            return np.array([]), np.array([]), t_final

        # --- CALCUL DYNAMIQUE DU DX RÉEL ---
        # Pour N particules dans un carré de côté L, dx = L / sqrt(N)
        n_fluid = len(pos_f)
        real_dx = L / np.sqrt(n_fluid)
        
        x_mid = L / 2.0
        band_width = real_dx * 1.5  # Largeur de bande proportionnelle au vrai dx
        condition_band = np.abs(pos_f[:, 0] - x_mid) < band_width

        # Sécurité : si la bande est vide, on prend toutes les particules (profil homogène en x)
        if not np.any(condition_band):
            condition_band = np.ones(len(pos_f), dtype=bool)

        y_sph = pos_f[condition_band, 1]
        u_sph = vel_f[condition_band, 0]

        sort_idx = np.argsort(y_sph)
        return y_sph[sort_idx], u_sph[sort_idx], t_final

    # Extraction des profils avec injection du dx propre à chaque cas
    y_sph1, u_sph1, t_f1 = extract_profile_from_series(vtk_files1, sim_dx=1.0)
    y_sph2, u_sph2, t_f2 = extract_profile_from_series(vtk_files2, sim_dx=1.0)
    y_sph3, u_sph3, t_f3 = extract_profile_from_series(vtk_files3, sim_dx=0.05)
    # DIAGNOSTIC
    print(f"Cas 1 (dx=1.0): {len(y_sph1)} points extraits, y range [{u_sph1.min():.4f}, {u_sph1.max():.4f}]")
    print(f"Cas 2 (dx=1.0): {len(y_sph2)} points extraits, y range [{y_sph2.min():.4f}, {y_sph2.max():.4f}]")

    # ============================================================================
    # TRACÉ DES GRAPHES COMPARATIFS MULTI-TEMPS
    # ============================================================================
    plt.figure(figsize=(11, 8))
    y_fine = np.linspace(0, L, 200)
    
    # Référence stationnaire asymptotique commune
    u_ana_steady = poiseuille_analytical(y_fine, 1e5, L, Fe_x, mu, nu)
    plt.plot(u_ana_steady, y_fine, 'k--', lw=1.5, alpha=0.5, label='Analytique Stationnaire ($t \\to \\infty$)')
    
    # --- Configuration 1 (t_f1 = 60s) ---
    u_ana_1 = poiseuille_analytical(y_fine, t_f1, L, Fe_x, mu, nu)
    plt.plot(u_ana_1, y_fine, color='green', linestyle='-', lw=2, label=f'Analytique Transitoire (t = {t_f1:.1f}s)')
    plt.scatter(u_sph1, y_sph1, color='green', marker='o', s=35, alpha=0.7, label='No MUSCL Euler NO RENORM (SPH)')
    
    # --- Configuration 3 (t_f3) ---
    if u_sph3 is not None and len(u_sph3) > 0:
        u_ana_3 = poiseuille_analytical(y_fine, t_f3, L, Fe_x, mu, nu)
        plt.plot(u_ana_3, y_fine, color='red', linestyle='-', lw=2, label=f'Analytique Transitoire (t = {t_f3:.1f}s)')
        plt.scatter(u_sph3, y_sph3, color='red', marker='x', s=45, alpha=0.7, label='No MUSCL Euler RENORM (SPH)')
        
    # --- Configuration 2 (t_f2 = 470s) ---
    if u_sph2 is not None and len(u_sph2) > 0:
        u_ana_2 = poiseuille_analytical(y_fine, t_f2, L, Fe_x, mu, nu)
        plt.plot(u_ana_2, y_fine, color='blue', linestyle='-', lw=2.5, label=f'Analytique Transitoire (t = {t_f2:.1f}s)')
        plt.scatter(u_sph2, y_sph2, color='blue', marker='s', edgecolor='k', s=40, alpha=0.8, label='MUSCL MLS1 RK2 (SPH)')

    plt.xlim(-0.1 * U0, U0 * 1.2)
    plt.ylim(-0.05 * L, L * 1.05)
    plt.xlabel('Vitesse longitudinale u [m/s]', fontsize=13)
    plt.ylabel('Hauteur du canal y [m]', fontsize=13)
    plt.title('Validation du profil de vitesse transitoire de Poiseuille', fontsize=14, fontweight='bold')
    plt.legend(loc='upper right', fontsize=10, frameon=True, shadow=True)
    plt.grid(True, which='both', linestyle='--', alpha=0.4)
    
    os.makedirs(VTK_DIR, exist_ok=True)
    plt.savefig(f'{VTK_DIR}/poiseuille_profile_validation.png', dpi=200, bbox_inches='tight')
    plt.show()

    # ============================================================================
    # EVALUATION DE L'ERREUR NUMÉRIQUE (Basée sur le cas 2)
    # ============================================================================
    errors_poiseuille = []
    if u_sph2 is not None and len(u_sph2) > 0:
        print(f"\n📊 Analyse d'erreur quantitative sur le Profil Transitoire à t = {t_f2:.2f}s:")
        print(f"{'Hauteur y (m)':>12} {'u_SPH (m/s)':>15} {'u_Ref (m/s)':>15} {'Erreur Rel.':>12}")
        print("-" * 60)
        
        u_ana_interp = poiseuille_analytical(y_sph2, t_f2, L, Fe_x, mu, nu)
        
        for y_p, u_p, u_a in zip(y_sph2, u_sph2, u_ana_interp):
            if u_a > 1e-2 * U0:
                err = abs(u_p - u_a) / U0
                errors_poiseuille.append(err)
                status = "✅" if err < 0.05 else "⚠️" if err < 0.15 else "❌"
                if np.random.rand() < 0.15:
                    print(f"{y_p:>12.4f} {u_p:>15.6f} {u_a:>15.6f} {err*100:>11.2f}% {status}")

        if len(errors_poiseuille) > 0:
            plt.figure(figsize=(6, 4))
            plt.plot(errors_poiseuille, y_sph2, 'b-', label='Erreur locale (MUSCL MLS1)')
            plt.axvline(x=0.05, color='g', linestyle='--', label='Seuil OK (5%)')
            plt.xlabel('Erreur relative normalisée / Umax [-]')
            plt.ylabel('Hauteur du canal y [m]')
            plt.title('Profil vertical de l\'erreur SPH')
            plt.legend(loc='lower right')
            plt.grid(True, alpha=0.3)
            plt.savefig(f'{VTK_DIR}/poiseuille_error_profile.png', dpi=150, bbox_inches='tight')
            plt.show()
            
            df_poiseuille = pd.DataFrame({
                'y_position_m': y_sph2,
                'u_velocity_sph_m_s': u_sph2,
                'u_velocity_analytical_m_s': u_ana_interp
            })
            df_poiseuille.to_csv(f'{VTK_DIR}/validation_poiseuille_data.csv', index=False)
            print(f"✅ Profil exporté sous: {VTK_DIR}/validation_poiseuille_data.csv")
            
            mean_err_p = np.mean(errors_poiseuille)
            max_err_p = np.max(errors_poiseuille)
            status_p = 'PASS' if max_err_p < 0.05 else 'WARN' if max_err_p < 0.15 else '❌ FAIL'
            
            poiseuille_report = f"""
  Poiseuille Flow (Transient Re=10):
    - Erreur moyenne normalisée: {mean_err_p*100:.2f}%
    - Erreur maximale localisée: {max_err_p*100:.2f}%
    - Status: {status_p}
            """
            print("\n✅ Analyse de Poiseuille intégrée.")
else:
    print("⏭️ Pas de fichiers VTK trouvés pour le cas d'application Poiseuille.")